In [30]:
# !conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia

# !conda install faiss-gpu=1.9.0 -c pytorch -c nvidia

In [31]:
import torch

print("📦 PyTorch version:", torch.__version__)
print("🚀 CUDA available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("🧠 GPU Name       :", torch.cuda.get_device_name(0))

📦 PyTorch version: 2.5.1
🚀 CUDA available : True
🧠 GPU Name       : NVIDIA RTX A4000


In [32]:
import faiss

print("📦 FAISS version :", faiss.__version__)

# Kiểm tra module FAISS-GPU có hoạt động không
try:
    res = faiss.StandardGpuResources()  # Nếu không lỗi là có GPU
    print("🚀 FAISS is using GPU ✅")
except Exception as e:
    print("❌ FAISS is NOT using GPU:", str(e))

📦 FAISS version : 1.9.0
🚀 FAISS is using GPU ✅


In [33]:
import os
import json

import pandas as pd
from ast import literal_eval

import torch
from torch.utils.data import DataLoader
from sentence_transformers.readers import InputExample
from sentence_transformers import SentenceTransformer, models, losses, util

from mteb import MTEB
from mteb.abstasks.TaskMetadata import TaskMetadata
from mteb.abstasks.AbsTaskRetrieval import AbsTaskRetrieval

from tqdm.autonotebook import tqdm

os.environ['WANDB_DISABLED'] = 'true'

In [34]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

## **I - Prepare Data**
---

In [35]:
import os
import requests
import zipfile

# Tạo thư mục nếu chưa có
os.makedirs("data", exist_ok=True)

# Đường dẫn tải và lưu
url = "https://huggingface.co/datasets/tmnam20/BKAI-Legal-Retrieval/resolve/main/archive.zip"
zip_path = "archive.zip"

# Tải file zip
print("⏬ Đang tải dữ liệu...")
response = requests.get(url)
with open(zip_path, "wb") as f:
    f.write(response.content)
print("✅ Tải xong!")

# Giải nén
print("📦 Đang giải nén...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("data")
print("✅ Giải nén thành công!")

# Xoá file zip nếu muốn
# os.remove(zip_path)

⏬ Đang tải dữ liệu...
✅ Tải xong!
📦 Đang giải nén...
✅ Giải nén thành công!


In [36]:
# !wget -q https://huggingface.co/datasets/tmnam20/BKAI-Legal-Retrieval/resolve/main/archive.zip
# !unzip -o -q archive.zip -d data

In [37]:
corpus_data = pd.read_csv('data/corpus.csv')
train_data  = pd.read_csv('data/train_split.csv', converters={'context': literal_eval})
test_data   = pd.read_csv('data/val_split.csv', converters={'context': literal_eval})

print(f"Train data: {len(train_data)}")
print(f"Test data : {len(test_data)}")

Train data: 89592
Test data : 29864


In [38]:
train_data['cid'] = train_data['cid'].apply(lambda x: [int(i) for i in x[1:-1].split()])
test_data['cid']  = test_data['cid'].apply(lambda x: [int(i) for i in x[1:-1].split()])

train_data.head()

,question,context,cid,qid
0,Liên đoàn Luật sư Việt Nam là tổ chức xã hội –...,[“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư...,[142820],72600
1,Tên hợp tác xã bị rơi vào trường hợp cấm thì c...,"[Tên hợp tác xã, liên hiệp hợp tác xã\n1. Tên ...","[27817, 72117]",147562
2,Tài xế lái xe ô tô khách 50 chỗ ngồi bao lâu t...,"[""1. Sử dụng lái xe bảo đảm sức khỏe theo tiêu...","[33215, 56201]",142107
3,Các bước chuẩn bị thủ thuật bó bột Cravate sẽ ...,[BỘT CRAVATE\n...\nIV. CHUẨN BỊ\n1. Người thực...,[148158],77353
4,Viên chức Hộ sinh hạng 4 có những nhiệm vụ gì ...,[Hộ sinh hạng IV - Mã số: V.08.06.16\n1. Nhiệm...,[188132],113090


In [39]:
data    = {'train': train_data, 'test': test_data}
samples = {'train': [], 'test': []}

for subset in ['train', 'test']:
    for _, row in data[subset].iterrows():
        question = row['question']
        context  = row['context']
        for c in context:
            samples[subset].append(InputExample(texts=[question, c]))

print(f"Train size: {len(samples['train'])}")
print(f"Test size : {len(samples['test'])}")

Train size: 89592
Test size : 29864


## **II - Fine-tune Sentence Transformers model**
---

In [40]:
os.makedirs('finetune_cache', exist_ok=True)
os.makedirs('finetune_output', exist_ok=True)

CACHE_DIR  = 'finetune_cache'
OUTPUT_DIR = 'finetune_output'

BATCH_SIZE = 128
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)

cuda


In [41]:
model_id = 'google-bert/bert-base-multilingual-cased'

word_embedding_model = models.Transformer(model_id, max_seq_length=512, cache_dir=CACHE_DIR)
pooling_model        = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True,
    pooling_mode_cls_token=False, 
    pooling_mode_max_tokens=False,
)

finetuned_model = SentenceTransformer(
    modules=[word_embedding_model, pooling_model], device=DEVICE, 
    cache_folder=CACHE_DIR
)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [42]:
train_dataloader = DataLoader(samples['train'], shuffle=True, batch_size=BATCH_SIZE,
                              pin_memory=True)

print(len(train_dataloader))

700


In [43]:
train_loss = losses.CachedMultipleNegativesRankingLoss(model=finetuned_model)

finetuned_model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=5,
    output_path=OUTPUT_DIR, 
    optimizer_params={'lr': 3e-5},
    show_progress_bar=True,
    use_amp=True,
    warmup_steps=100
)

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Step,Training Loss
500,0.385700
1000,0.108700
1500,0.073000
2000,0.051900
2500,0.039800
3000,0.033700
3500,0.029900


## **III - Evaluation with MTEB on the BKAI Legal Document Retrieval Dataset**
---

In [44]:
finetuned_model = SentenceTransformer(
    OUTPUT_DIR, device=DEVICE, 
    model_kwargs={'torch_dtype': 'float16'}
)

In [45]:
class BKAILegalDocRetrievalTask(AbsTaskRetrieval):
    # Metadata definition used by MTEB benchmark
    metadata = TaskMetadata(name='BKAILegalDocRetrieval',
                            description='',
                            reference='https://github.com/embeddings-benchmark/mteb/blob/main/docs/adding_a_dataset.md',
                            type='Retrieval',
                            category='s2p',
                            modalities=['text'],
                            eval_splits=['test'],
                            eval_langs=['vi'],
                            main_score='ndcg_at_10',
                            other_scores=['recall_at_10', 'precision_at_10', 'map'],
                            dataset={
                                'path'    : 'data',
                                'revision': 'd4c5a8ba10ae71224752c727094ac4c46947fa29',
                            },
                            date=('2012-01-01', '2020-01-01'),
                            form='Written',
                            domains=['Academic', 'Non-fiction'],
                            task_subtypes=['Scientific Reranking'],
                            license='cc-by-nc-4.0',
                            annotations_creators='derived',
                            dialect=[],
                            text_creation='found',
                            bibtex_citation=''
    )

    data_loaded = True # Flag

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        global corpus_data, data

        self.corpus        = {}
        self.queries       = {}
        self.relevant_docs = {}

        shared_corpus = {}
        for _, row in corpus_data.iterrows():
            cid_str                = f"c{row['cid']}"
            shared_corpus[cid_str] = {'text': row['text'], '_id': row['cid']} # Standard format for AbsTaskRetrieval

        for split in data:
            self.corpus[split]        = shared_corpus
            self.queries[split]       = {}
            self.relevant_docs[split] = {}

        for split in data:
            for i, row in data[split].iterrows():
                qid, cids = row['qid'], row['cid']
                question  = row['question']
                qid_str   = f'q{qid}'
                cids_str  = [f'c{cid}' for cid in cids]

                self.queries[split][qid_str] = question

                for cid_str in cids_str:
                    if cid_str not in self.relevant_docs[split]:
                        self.relevant_docs[split][qid_str] = {}
                    self.relevant_docs[split][qid_str][cid_str] = 1

        self.data_loaded = True

In [46]:
custom_task = BKAILegalDocRetrievalTask()
evaluation  = MTEB(tasks=[custom_task])
evaluation.run(finetuned_model, batch_size=BATCH_SIZE)

The `batch_size` argument is deprecated and will be removed in the next release. Please use `encode_kwargs = {'batch_size': ...}` to set the batch size instead.


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

Retrieval

- BKAILegalDocRetrieval, s2p

Batches: 100%|██████████| 91/91 [00:22<00:00,  3.99it/s]


[TaskResult(task_name=BKAILegalDocRetrieval, scores=...)]

## **IV - Retrieval**
---

In [47]:
import faiss

In [48]:
passages           = corpus_data['text'].tolist()
corspus_embeddings = finetuned_model.encode(
    passages, 
    batch_size=BATCH_SIZE,
    convert_to_numpy=True, 
    normalize_embeddings=True,
    show_progress_bar=True, 
    device=DEVICE
).astype(np.float32)

Batches: 100%|██████████| 2044/2044 [17:25<00:00,  1.96it/s]


In [ ]:
# d     = corspus_embeddings.shape[1] # 768
# index = faiss.IndexFlatIP(d)

# res   = faiss.StandardGpuResources()          # Use a single GPU
# index = faiss.index_cpu_to_gpu(res, 0, index) # Move index to GPU
# index.add(corspus_embeddings)                 # Add vectors to the index
# faiss.write_index(index, "data/legal_faiss.index")

RuntimeError: Error in void __cdecl faiss::write_index(const struct faiss::Index *,struct faiss::IOWriter *,int) at D:\bld\faiss-split_1734665785306\work\faiss\impl\index_write.cpp:858: don't know how to serialize this type of index

In [ ]:
# Bước 1: Tạo CPU index như ban đầu
d         = corspus_embeddings.shape[1]  # 768
cpu_index = faiss.IndexFlatIP(d)

# Bước 2: Tạo GPU index và thêm dữ liệu
res       = faiss.StandardGpuResources()
gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index)
gpu_index.add(corspus_embeddings)

In [ ]:
# Bước 3: Chuyển lại về CPU index
final_cpu_index = faiss.index_gpu_to_cpu(gpu_index)

# Bước 4: Ghi index vào file
faiss.write_index(final_cpu_index, "data/legal_faiss.index")

In [54]:
def search(model, query, index, k=10):
    query_embedding = model.encode(
        query, 
        convert_to_numpy=True, 
        normalize_embeddings=True,
    ).astype(np.float32).reshape(1, -1)

    scores, indices = index.search(query_embedding, k)
    hits = [{'score': scores[0][i], 'index': indices[0][i]} for i in range(len(scores[0]))]
    return hits

In [52]:
legal_index = faiss.read_index("data/legal_faiss.index")

In [57]:
query = "Hợp đồng lao động là gì?"

hits = search(finetuned_model, query, legal_index, k=10)
for rank, hit in enumerate(hits):
    print(f"Rank: {rank + 1}")
    print(f"Index: {hit['index']}, Score: {hit['score']}")
    print(passages[hit['index']])
    print("-" * 50)
    print('\n')

Rank: 1
Index: 55939, Score: 0.844379723072052
"Điều 13. Hợp đồng lao động
1. Hợp đồng lao động là sự thỏa thuận giữa người lao động và người sử dụng lao động về việc làm có trả công, tiền lương, điều kiện lao động, quyền và nghĩa vụ của mỗi bên trong quan hệ lao động.
Trường hợp hai bên thỏa thuận bằng tên gọi khác nhưng có nội dung thể hiện về việc làm có trả công, tiền lương và sự quản lý, điều hành, giám sát của một bên thì được coi là hợp đồng lao động.
2. Trước khi nhận người lao động vào làm việc thì người sử dụng lao động phải giao kết hợp đồng lao động với người lao động."
"Điều 15. Nguyên tắc giao kết hợp đồng lao động
1. Tự nguyện, bình đẳng, thiện chí, hợp tác và trung thực.
2. Tự do giao kết hợp đồng lao động nhưng không được trái pháp luật, thỏa ước lao động tập thể và đạo đức xã hội."
--------------------------------------------------


Rank: 2
Index: 55937, Score: 0.8325114250183105
"Điều 13. Hợp đồng lao động
1. Hợp đồng lao động là sự thỏa thuận giữa người lao động và